In [ ]:
# calibrate_camera.py
import cv2
import numpy as np
import glob
import os

IMG_DIR = "calibration/images"
OUT_DIR = "calibration/output"
CHECKERBOARD = (9, 6)   # inner corners (cols, rows) adjust to your printed pattern
SQUARE_SIZE_M = 0.02    # meters (e.g., 2 cm squares). Change to your printed checkerboard square size.

os.makedirs(IMG_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)

def capture_images():
    cap = cv2.VideoCapture(0)
    idx = 0
    print("Press SPACE to capture image, ESC to finish capture.")
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        cv2.imshow("Capture calibration images - press SPACE", frame)
        k = cv2.waitKey(1) & 0xFF
        if k == 32:  # SPACE
            idx += 1
            fname = f"{IMG_DIR}/img_{idx:02d}.jpg"
            cv2.imwrite(fname, frame)
            print("Saved", fname)
        elif k == 27:  # ESC
            break
    cap.release()
    cv2.destroyAllWindows()

def calibrate():
    # Prepare object points in world coordinates (0,0,0), (1,0,0), ...
    objp = np.zeros((CHECKERBOARD[1] * CHECKERBOARD[0], 3), np.float32)
    objp[:, :2] = np.mgrid[0:CHECKERBOARD[0], 0:CHECKERBOARD[1]].T.reshape(-1, 2)
    objp *= SQUARE_SIZE_M

    objpoints = []
    imgpoints = []
    images = glob.glob(os.path.join(IMG_DIR, "*.jpg"))
    if not images:
        print("No calibration images found; run capture_images() first.")
        return

    for fname in images:
        img = cv2.imread(fname)
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        ret, corners = cv2.findChessboardCorners(gray, (CHECKERBOARD[0], CHECKERBOARD[1]), None)
        if ret:
            objpoints.append(objp)
            # refine corner location
            corners2 = cv2.cornerSubPix(gray, corners, (11,11), (-1,-1),
                                         criteria=(cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 1e-6))
            imgpoints.append(corners2)
            cv2.drawChessboardCorners(img, (CHECKERBOARD[0], CHECKERBOARD[1]), corners2, ret)
            cv2.imshow("Corners", img)
            cv2.waitKey(100)
        else:
            print("Chessboard not found in", fname)
    cv2.destroyAllWindows()
    # calibrate camera
    ret, camera_matrix, dist_coeffs, rvecs, tvecs = cv2.calibrateCamera(objpoints, imgpoints, gray.shape[::-1], None, None)
    print("Calibration RMS error:", ret)
    print("Camera matrix:\n", camera_matrix)
    print("Dist coeffs:\n", dist_coeffs.ravel())
    np.save(os.path.join(OUT_DIR, "camera_matrix.npy"), camera_matrix)
    np.save(os.path.join(OUT_DIR, "dist_coeffs.npy"), dist_coeffs)
    # Save also object/image points for later homography if needed
    np.save(os.path.join(OUT_DIR, "objpoints.npy"), np.array(objpoints, dtype=object))
    np.save(os.path.join(OUT_DIR, "imgpoints.npy"), np.array(imgpoints, dtype=object))
    print("Saved calibration to", OUT_DIR)

if __name__ == "__main__":
    # Step A: capture images (optional)
    # capture_images()  # uncomment to capture images interactively
    calibrate()
